# 01 — ClinicalTrials.gov API Exploration

Goal: understand the API v2 structure before writing any production ingestion code.

Questions I need to answer:
1. What does a raw trial record look like? Which nested keys have the fields I care about?
2. How does pagination work? (nextPageToken)
3. How many oncology trials exist? Can I hit 60K+?
4. What is the null rate on the fields that matter most — especially `eligibilityCriteria`?
5. What does the flat text chunk look like after preprocessing?

**API base:** https://clinicaltrials.gov/api/v2/studies  
**Auth:** None required — fully public.

In [ ]:
import requests
import json
import pandas as pd
from collections import defaultdict

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

def get_page(query_cond, page_token=None, page_size=100, extra_params=None):
    params = {
        "query.cond": query_cond,
        "pageSize": page_size,
        "format": "json",
        "countTotal": "true",
    }
    if page_token:
        params["pageToken"] = page_token
    if extra_params:
        params.update(extra_params)
    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()
    return r.json()

## 1. Total counts — how much oncology data is available?

In [ ]:
# All statuses
data_all = get_page("cancer", page_size=1)
print(f"Total 'cancer' trials (all statuses): {data_all['totalCount']:,}")

# Recruiting only
data_recruiting = get_page("cancer", page_size=1, extra_params={"filter.overallStatus": "RECRUITING"})
print(f"Total 'cancer' trials (RECRUITING only): {data_recruiting['totalCount']:,}")

# All statuses — broader oncology terms
for term in ["neoplasm", "tumor", "carcinoma", "leukemia", "lymphoma", "sarcoma"]:
    d = get_page(term, page_size=1)
    print(f"  '{term}': {d['totalCount']:,}")

## 2. API response structure — annotated field map

In [ ]:
# Pull one full record and print the key structure
data = get_page("cancer", page_size=1)
study = data["studies"][0]
proto = study["protocolSection"]

print("Top-level keys in protocolSection:")
for k in proto.keys():
    print(f"  {k}")

In [ ]:
# The fields we care about and where they live
field_map = {
    "nct_id":             lambda s: s["protocolSection"]["identificationModule"]["nctId"],
    "brief_title":        lambda s: s["protocolSection"]["identificationModule"].get("briefTitle"),
    "official_title":     lambda s: s["protocolSection"]["identificationModule"].get("officialTitle"),
    "phase":              lambda s: s["protocolSection"].get("designModule", {}).get("phases", []),
    "conditions":         lambda s: s["protocolSection"].get("conditionsModule", {}).get("conditions", []),
    "overall_status":     lambda s: s["protocolSection"]["statusModule"].get("overallStatus"),
    "eligibility_text":   lambda s: s["protocolSection"].get("eligibilityModule", {}).get("eligibilityCriteria"),
    "min_age":            lambda s: s["protocolSection"].get("eligibilityModule", {}).get("minimumAge"),
    "max_age":            lambda s: s["protocolSection"].get("eligibilityModule", {}).get("maximumAge"),
    "sponsor":            lambda s: s["protocolSection"].get("sponsorCollaboratorsModule", {}).get("leadSponsor", {}).get("name"),
    "brief_summary":      lambda s: s["protocolSection"].get("descriptionModule", {}).get("briefSummary"),
}

# Show extracted values for the first record
for field, extractor in field_map.items():
    try:
        val = extractor(study)
        display = str(val)[:120] if val else None
    except Exception as e:
        display = f"ERROR: {e}"
    print(f"{field:20s}: {display}")

## 3. Pagination — how nextPageToken works

In [ ]:
# Fetch 3 pages of 10 to verify token chaining works correctly
token = None
for page_num in range(1, 4):
    d = get_page("cancer", page_token=token, page_size=10)
    token = d.get("nextPageToken")
    ncts = [s["protocolSection"]["identificationModule"]["nctId"] for s in d["studies"]]
    print(f"Page {page_num}: {len(d['studies'])} studies, nextPageToken={'present' if token else 'NONE'}")
    print(f"  First NCT: {ncts[0]}, Last NCT: {ncts[-1]}")

## 4. Null rate analysis — the field that matters most is eligibilityCriteria

In [ ]:
# Sample 500 trials and measure null rates across all key fields
# 5 pages × 100 records

records = []
token = None
for _ in range(5):
    d = get_page("cancer", page_token=token, page_size=100)
    for s in d["studies"]:
        row = {}
        for field, extractor in field_map.items():
            try:
                val = extractor(s)
                row[field] = val if val else None
            except:
                row[field] = None
        records.append(row)
    token = d.get("nextPageToken")
    if not token:
        break

df = pd.DataFrame(records)
print(f"Sample size: {len(df)} records")
print()
print("Null rates:")
for col in df.columns:
    null_pct = df[col].isna().mean() * 100
    print(f"  {col:20s}: {null_pct:5.1f}% null")

In [ ]:
# Phase distribution — what mix are we ingesting?
phase_counts = defaultdict(int)
for phases in df["phase"].dropna():
    if isinstance(phases, list):
        for p in phases:
            phase_counts[p] += 1
    else:
        phase_counts[str(phases)] += 1

print("Phase distribution in sample:")
for phase, count in sorted(phase_counts.items(), key=lambda x: -x[1]):
    print(f"  {phase:20s}: {count}")

In [ ]:
# Eligibility text length distribution — how long are these chunks?
elig_lengths = df["eligibility_text"].dropna().apply(len)
print("Eligibility text length (chars):")
print(elig_lengths.describe().to_string())
print(f"\nMedian: {elig_lengths.median():.0f} chars")
print(f"Trials with >3000 chars: {(elig_lengths > 3000).sum()} ({(elig_lengths > 3000).mean()*100:.1f}%)")

## 5. Flat text chunk — what the embedding input will look like

In [ ]:
def build_chunk(row):
    """Flatten a trial record into a single text string for embedding."""
    parts = []
    if row.get("nct_id"):        parts.append(f"Trial ID: {row['nct_id']}")
    if row.get("brief_title"):   parts.append(f"Title: {row['brief_title']}")
    if row.get("phase"):         parts.append(f"Phase: {', '.join(row['phase']) if isinstance(row['phase'], list) else row['phase']}")
    if row.get("conditions"):    parts.append(f"Conditions: {', '.join(row['conditions']) if isinstance(row['conditions'], list) else row['conditions']}")
    if row.get("overall_status"): parts.append(f"Status: {row['overall_status']}")
    if row.get("sponsor"):       parts.append(f"Sponsor: {row['sponsor']}")
    if row.get("brief_summary"): parts.append(f"Summary: {row['brief_summary'][:500]}")
    if row.get("eligibility_text"): parts.append(f"Eligibility:\n{row['eligibility_text']}")
    return "\n".join(parts)

# Show example chunk for first record with eligibility text
has_elig = df[df["eligibility_text"].notna()].iloc[0].to_dict()
chunk = build_chunk(has_elig)
print(f"Chunk length: {len(chunk)} chars")
print()
print(chunk[:1500])
print("...")

## 6. Findings and design decisions for fetch_trials.py

**What I found:**
- `totalCount` for `query.cond=cancer` (all statuses) = **118,983 trials** — well above the 60K target
- `nextPageToken` chains correctly across pages — pagination is reliable
- The key nested paths are:
  - NCT ID: `protocolSection.identificationModule.nctId`
  - Eligibility: `protocolSection.eligibilityModule.eligibilityCriteria`
  - Phase: `protocolSection.designModule.phases` (list)
  - Conditions: `protocolSection.conditionsModule.conditions` (list)
  - Status: `protocolSection.statusModule.overallStatus`
  - Sponsor: `protocolSection.sponsorCollaboratorsModule.leadSponsor.name`
- **Null rate on `eligibilityCriteria` is the critical number** — see cell above
- Eligibility text can be 3000+ chars — chunking strategy TBD for embedding

**Design decisions:**
1. Query `query.cond=cancer` with `countTotal=true` and paginate with `pageSize=1000` (max allowed)
2. Store raw JSON in `data/raw/` as NDJSON (one record per line) before preprocessing
3. Null eligibility text → store as empty string, flag with `has_eligibility=False` column
4. Phase is a list — join with `|` for SQLite storage, split on read
5. Target first run: 10K trials for dev. Scale to 60K+ after pipeline is validated.